In [24]:
# pip install은 GitHub Actions에서 처리하므로 노트북에선 생략
# (Colab에서 직접 돌릴 때만 아래 주석 해제)
# !pip install feedparser pandas

In [25]:
!pip install -q feedparser

In [26]:
!pip install -q --upgrade \
    "ccxt" \
    "requests==2.32.4" \
    "setuptools==81.0.0" \
    "jedi>=0.16"

In [27]:
import ccxt
import requests
import setuptools
import jedi

print("ccxt:", ccxt.__version__)
print("requests:", requests.__version__)
print("setuptools:", setuptools.__version__)
print("jedi:", jedi.__version__)

ccxt: 4.5.64
requests: 2.32.4
setuptools: 81.0.0
jedi: 0.20.0


In [28]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta, timezone
import calendar
from IPython.display import HTML # 클릭 가능한 링크 출력을 위해 추가

def get_recent_crypto_news(days=3):
    urls = {
        'CoinTelegraph': 'https://cointelegraph.com/rss',
        'CoinDesk': 'https://www.coindesk.com/arc/outboundfeeds/rss/'
    }

    news_data = []
    now = datetime.now(timezone.utc)
    time_threshold = now - timedelta(days=days)

    for source, url in urls.items():
        feed = feedparser.parse(url)

        for entry in feed.entries:
            if hasattr(entry, 'published_parsed') and entry.published_parsed:
                utctime = calendar.timegm(entry.published_parsed)
                pub_date = datetime.fromtimestamp(utctime, tz=timezone.utc)

                if pub_date >= time_threshold:
                    news_data.append({
                        '출처': source,
                        '제목': entry.title,
                        '링크': entry.link,
                        '발행시간(UTC)': pub_date.strftime('%Y-%m-%d %H:%M:%S')
                    })

    if news_data:
        df = pd.DataFrame(news_data)
        df = df.sort_values(by='발행시간(UTC)', ascending=False).reset_index(drop=True)
        return df
    else:
        return pd.DataFrame()

# 1. 데이터 수집
df_news = get_recent_crypto_news(days=3)

# 2. 클릭 가능한 링크로 변환하여 출력
if not df_news.empty:
    # 긴 주소 대신 깔끔하게 '🔗 보기' 버튼으로 변경 (새 창 열기 target="_blank" 적용)
    df_news['링크'] = df_news['링크'].apply(lambda x: f'<a href="{x}" target="_blank" style="color: #1a73e8; font-weight: bold; text-decoration: underline;">🔗 보기</a>')

    # 코랩에서 HTML 태그가 작동하도록 설정하여 출력
    display(HTML(df_news.to_html(escape=False, index=False)))
else:
    print("최근 3일 이내에 발행된 뉴스가 없습니다.")

# ── 텔레그램 자동 전송용 텍스트 요약 ──
if not df_news.empty:
    print("\n[코인 뉴스 TOP 5]")
    for _, row in df_news.head(5).iterrows():
        print(f"- {row['제목']}")
else:
    print("\n[코인 뉴스 TOP 5]: 해당 없음")


출처,제목,링크,발행시간(UTC)
CoinDesk,Tether posts $1.5 billion operating profit in Q2 as reserve buffer falls by half,🔗 보기,2026-07-31 18:24:55
CoinTelegraph,Here’s what happened in crypto today,🔗 보기,2026-07-31 18:09:51
CoinDesk,"The good and the bad of perps, according to crypto traders",🔗 보기,2026-07-31 17:53:59
CoinDesk,"Coldcard's $38 million (so far) exploit shakes faith in self-custody, may push investors to ETFs",🔗 보기,2026-07-31 17:07:30
CoinTelegraph,Ex-FTX users report funds being released in $900M distribution round,🔗 보기,2026-07-31 16:56:31
CoinTelegraph,Tether earns $1.5B in Q2 as US Treasury holdings fuel profits,🔗 보기,2026-07-31 16:25:47
CoinTelegraph,"Bybit adds tokenized Nvidia, Apple, Tesla stocks as loan collateral",🔗 보기,2026-07-31 16:23:21
CoinTelegraph,US Treasury yields rise as TIPS challenge the inflation narrative,🔗 보기,2026-07-31 16:14:19
CoinTelegraph,Bitcoin price sinks to 2-week lows as US stocks fail to copy Asia rebound,🔗 보기,2026-07-31 15:57:03
CoinDesk,"Quantum computing nears commercial breakthrough, IBM CEO says",🔗 보기,2026-07-31 15:39:30



[코인 뉴스 TOP 5]
- Tether posts $1.5 billion operating profit in Q2 as reserve buffer falls by half
- Here’s what happened in crypto today
- The good and the bad of perps, according to crypto traders
- Coldcard's $38 million (so far) exploit shakes faith in self-custody, may push investors to ETFs
- Ex-FTX users report funds being released in $900M distribution round


In [29]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta, timezone
import calendar
from IPython.display import HTML
import urllib.parse
import difflib  # 텍스트 유사도 비교를 위한 라이브러리

def get_smart_korean_stock_news(days=3):
    # 다양한 유형의 기사(시황, 분석, 특징주 등)를 포괄적으로 수집하기 위한 키워드
    keywords = "국내증시 OR 코스피 OR 코스닥 OR 주식시장"
    encoded_keywords = urllib.parse.quote(keywords)
    url = f"https://news.google.com/rss/search?q={encoded_keywords}&hl=ko&gl=KR&ceid=KR:ko"

    news_data = []
    now = datetime.now(timezone.utc)
    time_threshold = now - timedelta(days=days)

    # 필터링을 위한 카운터 및 키워드 정의
    ticker_count = 0
    max_ticker_articles = 3  # "단순 시황(얼마 돌파 등)" 기사는 최대 3개만 허용

    ticker_keywords = ['마감', '출발', '돌파', '포인트', '하락세', '상승세', '코스피', '코스닥', '보합']
    insight_keywords = ['전망', '분석', '이유', '배경', '특징주', '전략', '리포트', '진단', '과제', '수혜', '주목']

    print("구글 뉴스에서 한국 증시 소식을 수집 및 필터링 중...")
    feed = feedparser.parse(url)

    for entry in feed.entries:
        if hasattr(entry, 'published_parsed') and entry.published_parsed:
            utctime = calendar.timegm(entry.published_parsed)
            pub_date = datetime.fromtimestamp(utctime, tz=timezone.utc)

            # 3일 이내 기사만 대상
            if pub_date >= time_threshold:
                title = entry.title
                source = "기타"
                if " - " in title:
                    parts = title.rsplit(" - ", 1)
                    title = parts[0]
                    source = parts[1]

                # [필터 1] 제목 유사도 검사 (유사도 55% 이상이면 비슷한 뉴스로 판단하여 제외)
                is_duplicate = False
                for existing in news_data:
                    similarity = difflib.SequenceMatcher(None, title, existing['제목']).ratio()
                    if similarity > 0.55:
                        is_duplicate = True
                        break
                if is_duplicate:
                    continue  # 비슷한 내용의 기사는 패스

                # [필터 2] 단순 시황 기사 판별 및 개수 제한
                has_number = any(char.isdigit() for char in title)
                is_ticker = has_number and any(tk in title for tk in ticker_keywords)
                has_insight = any(ik in title for ik in insight_keywords)

                if is_ticker and not has_insight:
                    if ticker_count >= max_ticker_articles:
                        continue  # 단순 시황 기사가 이미 한도(3개)를 채웠다면 패스
                    ticker_count += 1
                    type_tag = "📊 단순시황"
                    priority = 2  # 정렬 우선순위 낮음
                else:
                    type_tag = "💡 인사이트/분석"
                    priority = 1  # 정렬 우선순위 높음

                news_data.append({
                    '우선순위': priority,
                    '구분': type_tag,
                    '출처': source,
                    '제목': title,
                    '링크': entry.link,
                    '발행시간(UTC)': pub_date.strftime('%Y-%m-%d %H:%M:%S')
                })

    if news_data:
        df = pd.DataFrame(news_data)
        # 인사이트 기사를 위로 올리고, 각각 그 안에서 최신순으로 정렬
        df = df.sort_values(by=['우선순위', '발행시간(UTC)'], ascending=[True, False]).reset_index(drop=True)
        # 내부 연산용 우선순위 컬럼은 시각적으로 제외
        df = df.drop(columns=['우선순위'])
        return df
    else:
        print("최근 3일 이내에 조건에 맞는 뉴스가 없습니다.")
        return pd.DataFrame()

# 1. 데이터 수집 및 스마트 필터링 실행
df_smart_news = get_smart_korean_stock_news(days=3)

# 2. 결과 출력 (클릭 가능한 링크 포함)
if not df_smart_news.empty:
    df_smart_news['링크'] = df_smart_news['링크'].apply(lambda x: f'<a href="{x}" target="_blank" style="color: #1a73e8; font-weight: bold; text-decoration: underline;">🔗 보기</a>')
    display(HTML(df_smart_news.to_html(escape=False, index=False)))

# ── 텔레그램 자동 전송용 텍스트 요약 ──
if not df_smart_news.empty:
    print("\n[한국 증시 뉴스 TOP 5]")
    for _, row in df_smart_news.head(5).iterrows():
        print(f"- {row['제목']}")
else:
    print("\n[한국 증시 뉴스 TOP 5]: 해당 없음")


구글 뉴스에서 한국 증시 소식을 수집 및 필터링 중...


구분,출처,제목,링크,발행시간(UTC)
💡 인사이트/분석,조선일보,"롤러코스피에 질린 개미들, 반등 시작하자 우르르 매도",🔗 보기,2026-07-31 15:40:00
💡 인사이트/분석,문화일보,국내증시 하락에…코스피 고점 이후 목표주가 하향 잇달아,🔗 보기,2026-07-31 15:12:10
💡 인사이트/분석,Investing.com 한국어,"시티, 투자자들이 코스피에서 ’저가 매수’ 모드로 전환했다고 밝혀",🔗 보기,2026-07-31 14:17:00
💡 인사이트/분석,v.daum.net,코스피 역대 최대 상승세... 하이닉스 상한가 마감,🔗 보기,2026-07-31 11:27:29
💡 인사이트/분석,v.daum.net,"코스피 급등 부른 뜻밖 배경 있었다…""악순환 종료"" 들썩",🔗 보기,2026-07-31 11:12:05
💡 인사이트/분석,문화일보,[이슈]코스피 17% 폭등? 데드캣 바운스 우려 이유,🔗 보기,2026-07-31 11:03:58
💡 인사이트/분석,뉴스핌,"[영상] '코스피 사상 최대 폭등'...삼전닉스, 지금 팔까요? 살까요?｜피플N이슈 장우진 대표편",🔗 보기,2026-07-31 09:58:00
💡 인사이트/분석,MBC 뉴스,뒤늦게 알려진 '미국발 폭탄'‥'미친 공포에 던졌는데‥' 절망,🔗 보기,2026-07-31 09:31:18
💡 인사이트/분석,연합뉴스,"'분노의 대반격' 코스피, 사상 최대 폭등…바닥 찍었나(종합)",🔗 보기,2026-07-31 07:56:27
💡 인사이트/분석,v.daum.net,코스피 급등에 개미 ‘대탈출’…“손실 만회하려 레버리지 베팅” 전망도,🔗 보기,2026-07-31 07:48:50



[한국 증시 뉴스 TOP 5]
- 롤러코스피에 질린 개미들, 반등 시작하자 우르르 매도
- 국내증시 하락에…코스피 고점 이후 목표주가 하향 잇달아
- 시티, 투자자들이 코스피에서 ’저가 매수’ 모드로 전환했다고 밝혀
- 코스피 역대 최대 상승세... 하이닉스 상한가 마감
- 코스피 급등 부른 뜻밖 배경 있었다…"악순환 종료" 들썩


In [30]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta, timezone
import calendar
from IPython.display import HTML
import urllib.parse
import difflib  # 영문 제목 유사도 비교용 라이브러리

def get_smart_us_stock_news(days=3):
    # 미국 시장을 포괄하는 핵심 매크로 키워드 조합 (URL 인코딩)
    keywords = '"stock market" OR "Wall Street" OR "S&P 500" OR "Nasdaq" OR "Dow Jones"'
    encoded_keywords = urllib.parse.quote(keywords)

    # 구글 뉴스 RSS (미국 설정 / 영어 뉴스 수집)
    url = f"https://news.google.com/rss/search?q={encoded_keywords}&hl=en-US&gl=US&ceid=US:en"

    news_data = []
    now = datetime.now(timezone.utc)
    time_threshold = now - timedelta(days=days)

    # 단순 시황 통제를 위한 카운터 및 키워드 설정
    ticker_count = 0
    max_ticker_articles = 3  # "단순 시황(Dow closes up 100pts 등)" 기사는 최대 3개만 허용

    # 단순 시황 판단을 위한 영문 키워드
    ticker_keywords = ['close', 'open', 'drop', 'rise', 'fall', 'gain', 'loss', 'hit', 'jump', 'slip', 'soar', 'tumble', 'point', 'rally', 'plummet']
    # 깊이 있는 분석 기사를 찾기 위한 영문 키워드
    insight_keywords = ['why', 'analysis', 'outlook', 'forecast', 'strategy', 'report', 'reason', 'insight', 'view', 'warn', 'expert', 'expect', 'opinion', 'fed', 'rate', 'inflation', 'recession', 'bubble', 'trend', 'future', 'investor']

    print("구글 뉴스에서 미국 증시 소식을 수집 및 필터링 중...")
    feed = feedparser.parse(url)

    for entry in feed.entries:
        if hasattr(entry, 'published_parsed') and entry.published_parsed:
            utctime = calendar.timegm(entry.published_parsed)
            pub_date = datetime.fromtimestamp(utctime, tz=timezone.utc)

            # 3일 이내 기사만 대상
            if pub_date >= time_threshold:
                title = entry.title
                source = "Unknown"
                if " - " in title:
                    parts = title.rsplit(" - ", 1)
                    title = parts[0]
                    source = parts[1]

                # [필터 1] 제목 유사도 검사 (중복 뉴스 원천 차단)
                is_duplicate = False
                for existing in news_data:
                    similarity = difflib.SequenceMatcher(None, title, existing['제목 (Title)']).ratio()
                    if similarity > 0.55:  # 유사도가 55%를 넘어가면 동일 주제로 판별
                        is_duplicate = True
                        break
                if is_duplicate:
                    continue

                # [필터 2] 영문 텍스트 패턴 분석 기반 시황 vs 인사이트 분류
                title_lower = title.lower()
                has_number = any(char.isdigit() for char in title)
                # 제목에 숫자가 있고, 등락 관련 단어가 포함되어 있다면 단순 시황으로 1차 의심
                is_ticker = has_number and any(tk in title_lower for tk in ticker_keywords)
                # 원인 분석이나 전망 키워드가 포함되어 있는지 확인
                has_insight = any(ik in title_lower for ik in insight_keywords)

                if is_ticker and not has_insight:
                    if ticker_count >= max_ticker_articles:
                        continue  # 단순 지수 중계 기사 수집 한도를 초과하면 제외
                    ticker_count += 1
                    type_tag = "📊 단순시황 (Market Update)"
                    priority = 2  # 아래쪽에 배치
                else:
                    type_tag = "💡 인사이트/분석 (Insight)"
                    priority = 1  # 최상단에 배치

                news_data.append({
                    '우선순위': priority,
                    '구분': type_tag,
                    '출처 (Source)': source,
                    '제목 (Title)': title,
                    '링크 (Link)': entry.link,
                    '발행시간(UTC)': pub_date.strftime('%Y-%m-%d %H:%M:%S')
                })

    if news_data:
        df = pd.DataFrame(news_data)
        # 인사이트 기사를 무조건 위로 배치하고, 그 안에서 최신 시간순 정렬
        df = df.sort_values(by=['우선순위', '발행시간(UTC)'], ascending=[True, False]).reset_index(drop=True)
        df = df.drop(columns=['우선순위'])
        return df
    else:
        print("최근 3일 이내에 조건에 맞는 미국 증시 뉴스가 없습니다.")
        return pd.DataFrame()

# 1. 데이터 수집 및 스마트 필터링 실행
df_us_smart_news = get_smart_us_stock_news(days=3)

# 2. 결과 출력 (클릭 가능한 링크 적용)
if not df_us_smart_news.empty:
    df_us_smart_news['링크 (Link)'] = df_us_smart_news['링크 (Link)'].apply(lambda x: f'<a href="{x}" target="_blank" style="color: #1a73e8; font-weight: bold; text-decoration: underline;">🔗 보기</a>')
    display(HTML(df_us_smart_news.to_html(escape=False, index=False)))

# ── 텔레그램 자동 전송용 텍스트 요약 ──
if not df_us_smart_news.empty:
    print("\n[미국 증시 뉴스 TOP 5]")
    for _, row in df_us_smart_news.head(5).iterrows():
        print(f"- {row['제목 (Title)']}")
else:
    print("\n[미국 증시 뉴스 TOP 5]: 해당 없음")


구글 뉴스에서 미국 증시 소식을 수집 및 필터링 중...


구분,출처 (Source),제목 (Title),링크 (Link),발행시간(UTC)
💡 인사이트/분석 (Insight),WSJ,"Apple Falls, But Amazon, Other Stocks Push Nasdaq Composite Higher",🔗 보기,2026-07-31 18:58:00
💡 인사이트/분석 (Insight),Investing.com,Wall Street climbs as Amazon soothes AI jitters By Reuters,🔗 보기,2026-07-31 18:30:30
💡 인사이트/분석 (Insight),Barron's,These Were the Best and Worst S&P 500 Stocks in July,🔗 보기,2026-07-31 18:16:00
💡 인사이트/분석 (Insight),Fox Business,Y'all Street launches: Texas Stock Exchange goes live in Dallas,"<a href=""https://news.google.com/rss/articles/CBMimgFBVV95cUxOcG4yeFdXdl9WOGdVdFlBTTlyOE1PUHVwdmpWd1VPTEZWbGlTeEJGVHFjSnZBSlhnZGxFMlhBQVJwSjREYWtTVWdmYlpkb0lWN09fcC0xNVBXaGxkU2l3RjMzV0l4emVlS3JlZF9scTNYem96Y203dHJfMTlJdmpWaS1hdHM2UmFuMF9fbmFIVGpFaVkyS0JpRE1B0gGfAUFVX3lxTE9ZdWFNazY2eWJvcW5od05UTFUwT3AzQ3lOVkdmTFl3R01IWG5uM29QYUp1MEhpOUlLMXNSYkcxbHA0cHhhenlDNW5GYjBXMzhWd2JTSWN0cEdONFhjOW83a1p4SWhtd0p0OHZCMUg5enhsVnJoOFNkNW9tcUxmWTg5WE5kWFJZNV9SS2ZTTGFQbS1TQW5oZG80MmNjZ3hqaw?oc=5"" target=""_blank"" style=""color: #1a73e8; font-weight: bold; text-decoration: underline;"">🔗 보기",2026-07-31 18:14:00
💡 인사이트/분석 (Insight),Yahoo Finance,1 of Wall Street’s Favorite Stocks Worth Your Attention and 2 We Question,🔗 보기,2026-07-31 18:11:57
💡 인사이트/분석 (Insight),CNBC,The S&P 500 is stuck at a key battleground level. This obscure index could determine the market’s next move,"<a href=""https://news.google.com/rss/articles/CBMiswFBVV95cUxOVW1mbmZheTI4aEkzTjc1MVBidWdGLVZRZ1ZudjhWaTRkN3BKRHEzWW1kbkVDWmxXVVJHLVRhU3VFS2xlUF84TXB3cHdSMWJUcWNpRV9mMkFfal9Sd29vNjdJU21TYkNMQVN0cUdBakRscExDTE5wbHE0b0tkM3laUTg2RlNzZWQ0cmtqTEhIcnJXRTJVNXc1clhRQklLVDRkbjNDc3FOb3pueUM4MklhSUktb9IBuAFBVV95cUxNT2QwQUtQU2NNaDRKNGpBMnFkdG5nLXBWS2VXNXo3azhkUFR6U25NTC05cVh1UHVVcFhTeFRISW1xWWFBMDI3aEZIN0VUa0dSTV9xck8tcTB4TXZTSEtkbE9nMzBVOV82Vlh1LVh1OFdCYWktRFlMZThuNDE3WDEwRFhtdUlGeGNhZlQzdG83V1UyeThHVnY2eGU0ZW55Q0x4U1hQTEhhLXQ5eUhsTm5Pc0JxRmZKRF9V?oc=5"" target=""_blank"" style=""color: #1a73e8; font-weight: bold; text-decoration: underline;"">🔗 보기",2026-07-31 18:07:36
💡 인사이트/분석 (Insight),Reuters,Wall St struggles for direction as rate uncertainty offsets Amazon jump,🔗 보기,2026-07-31 17:53:06
💡 인사이트/분석 (Insight),GlobeNewswire,ADMA Biologics (NASDAQ: ADMA) Investigated for Potential,🔗 보기,2026-07-31 17:20:21
💡 인사이트/분석 (Insight),Barron's,Wall Street Braces for Big Stock Moves That Could Roil Sleepy August Markets,🔗 보기,2026-07-31 17:16:00
💡 인사이트/분석 (Insight),Barron's,Flying Blind: Stock Market Survives White-Knuckle Week Despite Meta and Warsh Meltdowns,🔗 보기,2026-07-31 17:13:00



[미국 증시 뉴스 TOP 5]
- Apple Falls, But Amazon, Other Stocks Push Nasdaq Composite Higher
- Wall Street climbs as Amazon soothes AI jitters By Reuters
- These Were the Best and Worst S&P 500 Stocks in July
- Y'all Street launches: Texas Stock Exchange goes live in Dallas
- 1 of Wall Street’s Favorite Stocks Worth Your Attention and 2 We Question
